# 循环神经网络

目前为止，我们的简单文本生成模型在通往真正的语言理解道路上存在两个瓶颈：

* **无法理解语序**：对于**词袋**来说，“老鼠爱大米”和“大米爱老鼠”生成的上下文特征几乎是一样的。词语出现的先后顺序这一重要信息被丢失了。

* **缺乏长程记忆**：模型的视野被限制在一个狭小的**上下文窗口**（Context Window）内。超出了这个范围的文字，模型就完全无法建立起词义间的关联。比如这句歌词：“我爱你就像老鼠爱大米”。在上下文尺寸为 5 的情况下，就无法学习到“我爱你”和“老鼠爱大米”之间的关联。

---

怎样让机器拥有“长期记忆”，并不是一个新问题。

1986 年：语言学家**埃尔曼**（Elman）提出了**简单循环网络结构**（Simple RNN）。埃尔曼提出将**隐状态**（Hidden State）反馈到下一个时刻，这成为了现代 **循环神经网络**（Recurrent Neural Network, RNN）的标准范式：模型不仅看当前输入，也参考过去的自己。

所谓**隐状态**，就是在网络模型中，输入数据经过层层变换，准备进入最后的输出层时的状态。隐状态保存了网络模型到此刻为止，学习到的全部知识，就是所谓的**时间记忆**（temporal memory）。

回头看我们的词向量网络模型，只有两层：嵌入层和输出层。其中嵌入层的输出就是隐状态。

简单地讲，**循环神经网络**就是将上一次迭代进入输出层前的隐状态，连接到本次迭代输入层（嵌入层）后的词向量中。这样，一次迭代将包括四个步骤：

* 嵌入层对输入数据进行初步学习，生成词向量；
* 连接词向量和上次迭代的隐状态；
* 对叠加了时间记忆的词向量进一步学习，生成本次迭代的隐状态；
* 输出层生成预测值。

最终的输出数据包括**预测值**和**隐状态**两个部分。预测值用于损失函数开始梯度计算和反向传播；隐状态则参与到下一次迭代。

<div style="text-align: center;">
  <img src="images/recurrent.png" width="720">
  <div>图例：循环神经网络</div>
</div>

这里：

* $x$：（输入）特征值；
* $h$：（上一次迭代）隐状态；
* $h^{\prime}$：（输出）隐状态；
* $p^{\prime}$：（输出）预测值。

In [1]:
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import requests

In [2]:
np.random.seed(42)

## 张量

### 反向函数

张量的反向函数是反向传播的最主要载体：我们在反向函数中遍历计算图中所有父节点，计算父节点梯度。

我们一直使用非常简洁的**递归遍历算法**。这在计算图中所有节点排成一条直线或者树状时有效，每个节点只会被访问一次。而循环网络模型恰恰打破了这个前提。RNN 里，隐状态 $h$ 每读一个字符就被更新一次，同一个计算逻辑会被反复调用几十次。于是在计算图里，一个节点可以被多次访问，梯度被重复计算。

解决办法是换用**拓扑排序算法**：先把计算图里的所有节点按照**依赖顺序**排列，保证每个节点都排在它的所有父节点之后。这样我们从这个节点序列的末尾（损失值张量）开始，倒序依次访问每个节点，计算梯度。每个节点恰好被访问一次，梯度计算既不会重复也不会遗漏。

### 加法

RNN 的一个重要步骤就是将上一次迭代的隐状态 $h$ 加到新的特征嵌入向量上。这不是简单地把两个向量相加，而是需要像嵌入层、平均池化层那样处理前向传播中构建计算图，和反向传播中计算梯度的任务。

我们的解决方案是扩展张量的**加法运算**：在进行加法运算的同时，实现计算结果张量的梯度函数 gradient_fn 和父节点列表 parents。

In [3]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        topo = []
        visited = set()
        stack = [(self, False)]

        while stack:
            node, expanded = stack.pop()
            if node in visited:
                continue

            if expanded:
                visited.add(node)
                topo.append(node)
            else:
                stack.append((node, True))
                for p in node.parents:
                    if p not in visited:
                        stack.append((p, False))

        self.grad = np.ones_like(self.data)
        for t in reversed(topo):
            if t.gradient_fn is not None:
                t.gradient_fn()

        for t in topo:
            t.gradient_fn = lambda: None
            t.parents = set()

    def __add__(self, other):
        p = Tensor(self.data + other.data)

        def gradient_fn():
            self.grad += p.grad
            other.grad += p.grad

        p.gradient_fn = gradient_fn
        p.parents = {self, other}
        return p

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

### 抽象类

In [4]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

### 字符编码数据集

前面几章里，我们一直采用**下一个字符**的**单热编码**作为样本的标签值。

而 RNN 是对一个样本逐字符处理的。例如我们的上下文尺寸是 32，那么每个样本是 32 个字符。RNN 从第一个字符开始，依次进行 32 次预测，每次要带上之前一次的隐状态。因此一共预测出了 32 个字符，正好对应后移一位的 32 个字符。

因此，在 RNN 模型里，我们将采用：

* **x**（特征值）：`tokens[i: i + self.context_size]`
* **y**（标签值）：`tokens[i + 1: i + self.context_size + 1]`

就是说：如果特征值是文本位置 1-32 的字符，那么标签值就是位置 2-33 的字符，分别对应 RNN 32 次预测的结果。

In [5]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(tokens[i + 1: i + self.context_size + 1])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

## 模型

### 抽象层

In [6]:
class Layer(ABC):

    def __call__(self, *args):
        return self.forward(*args)

    @abstractmethod
    def forward(self, *args):
        pass

    @property
    def parameters(self):
        return []

### 线性层

In [7]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

### 组合抽象层

这里我们将顺序层抽象出**组合抽象层**（Composite Layer），只保留子层容器的功能。

In [8]:
class Composite(Layer, ABC):

    def __init__(self, layers):
        super().__init__()
        self.layers = list(layers)

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

### 顺序层

顺序层改为从组合抽象层继承，功能不变。

In [9]:
class Sequential(Composite):

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

### 嵌入层

In [10]:
class Embedding(Layer):

    def __init__(self, vocab_size, embedding_size, std=0.02):
        super().__init__()
        self.weight = Tensor(np.random.randn(vocab_size, embedding_size) * std)

    def forward(self, x: Tensor):
        p = Tensor(self.weight.data[x.data])

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

## 激活函数

### 线性整流函数（ReLU）

In [11]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 双曲正切函数（Tanh）

In [12]:
class Tanh(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.tanh(x.data))

        def gradient_fn():
            x.grad += a.grad * (1 - a.data ** 2)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

## 损失函数

### 抽象类

In [13]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 交叉熵损失函数

数据集不再返回单热编码作为标签值，而是和特征值同样格式的**索引编码序列**。

我们因此升级交叉熵损失函数，通过判断标签值的格式，同时支持**单热编码**和**索引编码序列**两种数据集。

In [14]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        if y.data.ndim == softmax.ndim:
            labels = y.data
        else:
            labels = np.zeros_like(softmax)
            rows = np.arange(y.data.size)
            labels[rows, y.data] = 1

        log = np.log(np.clip(softmax, 1e-10, 1.0))
        ce = Tensor(-np.mean(np.sum(labels * log, axis=-1)))

        def gradient_fn():
            p.grad += (softmax - labels) / labels.shape[0]

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

## 优化器（随机梯度下降）

In [15]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

## 循环神经网络

### 循环单元

RNN 的核心是**循环单元**（Cell）。它只做一件事：把**当前读到的字符** $x_t$ 和**之前的记忆** $h_{t-1}$ 融合成**新的记忆** $h_t$。

它内部有三个子层：

* **input**（输入线性层）：处理当前读到的字符 $x_t$；
* **hidden**（隐状态线性层）：处理之前的记忆 $h_{t-1}$；
* **tanh**（激活函数）：将上面两者相加的结果压缩到 $(-1, 1)$，得到新的记忆 $h_t$。

用公式表示就是：

$$
h_t = \tanh(W_{in} \, x_t + W_{hidden} \, h_{t-1} + b)
$$

这里 $h_{t-1}$ 是上一个**时间步**（Time Step）的记忆，$h_t$是更新后的记忆。记忆就这样一个字符、一个字符地往后传，**循环**（Recurrent）这个名字也由此而来。

In [16]:
class Cell(Composite):

    def __init__(self, in_size, out_size):
        self.input = Linear(in_size, out_size)
        self.hidden = Linear(out_size, out_size)
        self.tanh = ReLU()

        super().__init__([self.input,
                          self.hidden,
                          self.tanh])

    def forward(self, x: Tensor, h: Tensor):
        p = self.input(x)
        h = self.hidden(h)
        return self.tanh(p + h)

### 循环网络

把循环单元应用到一个完整的神经网络，就得到了 RNN。它处理一个完整的标签值（一段字符序列）的方式是：

* **embedding**（嵌入）：每个字符经过嵌入层转换成嵌入向量；
* **cell**（循环）：和之前记忆融合，得到新的记忆；
* **output**（输出）：新的记忆经过输出层，得到**下一个字符**的预测概率。

推理函数 forward 里的 for 循环，就是把特征值的字符序列**逐个字符**送入**同一个**循环单元 cell，并把新的记忆传给下一个时间步。这样，每个时间步都在训练同一套循环单元参数。

#### 随时间反向传播

这种**共享同一个单元、随时间步展开**的算法结构，在反向传播时也会造成梯度沿着时间步一路回传到最开始。称为**随时间反向传播**（Backpropagation Through Time, BPTT）。

``💡 梯度计算要穿过几十甚至上百个时间步，特别容易造成梯度消失或者爆炸，这也是 RNN 后来被 Transformer 取代的原因之一。``

In [17]:
class RNN(Composite):

    def __init__(self, vocab_size, embedding_size):
        self.embedding_size = embedding_size

        self.embedding = Embedding(vocab_size, embedding_size)
        self.cell = Cell(embedding_size, embedding_size * 2)
        self.output = Linear(embedding_size * 2, vocab_size)

        super().__init__([self.embedding,
                          self.cell,
                          self.output])

    def step(self, token: Tensor, h: Tensor):
        p = self.embedding(token)
        h = self.cell(p, h)
        return self.output(h), h

    def forward(self, x: Tensor, h: Tensor = None):
        batch_size, sequence_len = x.data.shape
        if h is None:
            h = Tensor(np.zeros((batch_size, self.embedding_size * 2)))

        ps = []
        for s in range(sequence_len):
            p, h = self.step(Tensor(x.data[:, s]), h)
            ps.append(p)

        return ps, h

## 训练器

### 抽象类

In [18]:
class Trainer(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

### 循环网络训练器

#### 训练函数

RNN 的随时间步训练和之前的多层神经网络略有不同：模型一次返回的是**一串预测**（每个时间步一个），而不是一个预测。所以我们要把一次推理的每个时间步的损失累加起来，然后做一次反向传播。

这里我们就用上了张量里我们扩展的加法 `__add__`：

`loss += self.loss_fn(...)`

这样我们在累加每个时间步损失的同时，也将每个时间步的计算图串联起来。通过一次反向传播，梯度就会沿着计算图，穿过每个时间步，把梯度传回每一个参数。

#### 生成函数

generate 也变成了逐字符生成：

* 先让模型把提示词 prompt 逐个处理，更新记忆；
* 根据当前记忆预测下一个字符，再次更新记忆；
* 重复第二步，并将每一步的预测串联成最终的输出文字。

In [19]:
class RNNTrainer(Trainer):

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction, _ = self.layer(feature)
                loss = Tensor(0.0)
                for t in range(len(prediction)):
                    loss += self.loss_fn(prediction[t], Tensor(label.data[:, t]))
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction, _ = self.layer(feature)
        loss = Tensor(0.0)
        for t in range(len(prediction)):
            loss += self.loss_fn(prediction[t], Tensor(label.data[:, t]))
        return prediction, loss

    def generate(self, dataset, prompt, steps=512):
        tokens = dataset.encode(prompt)

        h = Tensor(np.zeros((1, self.layer.embedding_size * 2)))
        for token in tokens:
            p, h = self.layer.step(Tensor([token]), h)

        for _ in range(steps):
            exp = np.exp(p.data[0] - np.max(p.data[0]))
            probs = exp / np.sum(exp)
            token = np.random.choice(len(probs), p=probs)
            tokens.append(token)
            p, h = self.layer.step(Tensor([token]), h)

        return dataset.decode(tokens)

## 超参数

### 学习率

In [20]:
LEARNING_RATE = 0.001

### 批大小

In [21]:
BATCH_SIZE = 4

### 轮数

In [22]:
EPOCHS = 2

### 上下文尺寸

In [23]:
CONTEXT_SIZE = 32

### 嵌入尺寸

In [24]:
EMBEDDING_SIZE = 16

## 数据准备

In [25]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

## 建模

我们已经把 RNN 的所有子层在循环单元 Cell 和 网络模型 RNN 中实现，因此只需要创建一个 RNN 的实例即可。

In [26]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = RNN(dataset.vocab_size, EMBEDDING_SIZE)
loss_fn = CELoss()
optimizer = SGDOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = RNNTrainer(model, loss_fn, optimizer)

## 训练

In [27]:
trainer.train(dataset, EPOCHS)

## 评估

In [28]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{len(prediction)} steps, each {prediction[0].data.shape}\nloss:\t{loss}')

prediction:	32 steps, each (6970, 65)
loss:	Tensor(69.57489934391883)


## 推理

In [29]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
Has that she you by sentins gewer.

GLA::
He spadio pill; you to wither wall o walchori, giand, an un't I eprreer me is, Thine she of this ilfe's, my it letjeand the freal,
Gor:
she llat him'd,
This hol.

DenETCAROUENA:
you geang,

CKURTENCIO:
Hes mabe ticl fee met a mave nebe ces to beglove do and, To he mand's heard hat you vealnemers moth tich farcker as yous fors me, gratenielods alr wold, then in mull cester ole a weld sin, a'd bedom duop lastsante leef!

LURETRA:
To on dionl:
And you it wer wellm hin


这一次，我们终于让网络模型**读懂了词序**：通过**记忆**（隐状态）机制，RNN 一个字符、一个字符地处理输入，每处理一个字符就更新一次记忆。

从上面的推理结果就可以看出，输出结果已经有单词的影子了。我们已经可以辨认出一些单词，这和之前完全是一堆乱字符的情况相比，是很大的进步。也说明 RNN 的确有效。

## 课后练习

把 Cell 里的激活函数从 Tanh 换成 ReLU，再训练一次，观察生成结果有什么不同？结合 Tanh 的输出结果范围 `(-1, 1)` 考虑一下原因。